# FIAT Calibration with Ostrich Constraints

This notebook demonstrates how to add **optimization constraints** to a FIAT calibration workflow using Ostrich's General-purpose Constrained Optimization Platform (GCOP).

## What are constraints?

Constraints define **bounds on model performance metrics** that, when violated, add a penalty to the objective function. This allows you to:

- Keep bias within acceptable limits (e.g., |PBIAS| < 25%)
- Enforce minimum model quality thresholds
- Combine multiple performance criteria beyond the primary objective

## How do constraints work in Ostrich?

Each constraint references a **response variable** (a scalar metric computed by `eval.py`). The constraint defines:
- **lower bound** (`g_min`): if the metric value `g < g_min`, a penalty is added
- **upper bound** (`g_max`): if the metric value `g > g_max`, a penalty is added
- **cost factor** (`CF`): multiplier that converts the violation into a penalty cost

The penalty for a single constraint is:
```
P = CF * (g_min - g)   if g < g_min
P = CF * (g - g_max)   if g > g_max
P = 0                   otherwise
```

The total penalty `P_TOTAL` is the sum of all constraint penalties, combined with the system cost via the Additive Penalty Method (APM):
```
F = C_SYS + P_TOTAL
```

## Constraint groups

Constraints follow the same group structure as objective functions:
- **`flux`**: Metrics computed per station (HydroErr metrics or user-defined callables)
- **`helper`**: Intermediate constraint values (not written to CSV, not seen by Ostrich directly)
- **`custom`**: Expressions referencing constraint helpers (written to CSV, used by Ostrich)

## This example

We calibrate the Wolf Creek Research Basin MESH model with:
- **Objective function**: Minimize negative NSE (maximize NSE)
- **Constraint**: PBIAS must be within +/-25% (Moriasi et al., 2007 guideline for streamflow)

In [ ]:
import numpy as np
import xarray as xr
import fiatmodel

## 1. Define parameter bounds

Same calibration parameters as the base Wolf Creek example.

In [ ]:
class_dict_bounds = {
    6: [
        {
            'class': 'needleleaf',
            'lnz0': [0.5, 4.0],
        },
        {
            'class': 'broadleaf',
            'lnz0': [0.2, 0.8]
        },
    ]
}

hydrology_dict_bounds = {
    15: {
        'zpls': [0.02, 0.6],
    },
}

routing_dict_bounds = {
    6: {
        'flz': [1e-6, 1e-3, "log10"],
    },
}

## 2. Define the PBIAS metric

PBIAS (Percent Bias) is not included in HydroErr, so we define it explicitly.
This callable is passed directly to FIAT as a constraint metric key.

In [ ]:
def pbias(sim, obs):
    return 100.0 * (obs - sim).sum() / obs.sum()

## 3. Load observations

In [ ]:
obs_obj = xr.open_dataset('./wolf-creek-research-basin/wolf-creek-gauge-data.nc')

## 4. Build the Calibration object with constraints

The key addition is the `constraints` key in `calibration_config`. Each constraint metric entry is a dictionary with:
- `lower`: lower bound
- `upper`: upper bound
- `cost_factor`: penalty multiplier
- `expressions`: list of numexpr expressions (same syntax as objective functions)

### Constraint definition in this example:

| Constraint | Metric | Bounds | Cost Factor | Rationale |
|---|---|---|---|---|
| PBIAS bound | pbias (user-defined) | [-25, +25] | 100 | Moriasi et al. "satisfactory" threshold |

In [ ]:
c = fiatmodel.Calibration(
    calibration_software='ostrich',
    model_software='mesh',
    calibration_config={
        'instance_path': './wolf-creek-calibration-with-constraints/',
        'random_seed': 10,
        'algorithm': 'DDS',
        'algorithm_specs': {
            'PerturbationValue': 0.2,
            'MaxIteration': 10_000,
            'UseRandomParamValue': None,
        },
        'spinup_start': '1980-01-01 12:00:00',
        'dates': [
            {
                'start': '1980-01-01 23:00:00',
                'end': '1980-01-02 23:00:00',
            },
        ],
        'objective_functions': {
            'flux': {
                'QO': {
                    'nse': ['-1 * alaska_72'],
                },
            },
        },
        'constraints': {
            'flux': {
                'QO': {
                    pbias: {
                        'lower': -25.0,
                        'upper': 25.0,
                        'cost_factor': 100,
                        'expressions': ['alaska_72'],
                    },
                },
            },
        },
    },
    model_config={
        'instance_path': './wolf-creek-research-basin/',
        'parameter_bounds': {
            'class': class_dict_bounds,
            'hydrology': hydrology_dict_bounds,
            'routing': routing_dict_bounds,
        },
        'executable': 'sa_mesh',
    },
    observations=[
        {
            "name": "alaska_72",
            "type": "QO",
            "timeseries": obs_obj['discharge'].isel(gauge_name=2).to_series(),
            "unit": "m^3/s",
            "scale_factor": 1,
            "offset_factor": 0,
            "computational_unit": "subbasin",
            "computational_unit_id": 38,
            "freq": "1h",
        },
    ],
)

## 5. Prepare the calibration instance

`c.prepare()` generates all files needed by Ostrich, including:
- `ostIn.txt` with `BeginConstraints/EndConstraints` block
- `eval.json` with constraint definitions for `eval.py`
- Constraint CSV output paths registered in `BeginResponseVars`

In [ ]:
c.prepare(output_path='./wolf-creek-calibration-with-constraints/')

## 6. Verify the generated files

Let's inspect the generated `ostIn.txt` to confirm the constraints are correctly rendered.

In [ ]:
with open('./wolf-creek-calibration-with-constraints/ostIn.txt', 'r') as f:
    content = f.read()
print(content)

### Check: BeginConstraints block

You should see a `BeginConstraints ... EndConstraints` block with one entry:

```
BeginConstraints
  constraint_flux_QO_pbias_1  general  10  -25.0  25.0  constraint_flux_QO_pbias_1
EndConstraints
```

In [ ]:
assert 'BeginConstraints' in content, "BeginConstraints block not found!"
assert 'EndConstraints' in content, "EndConstraints block not found!"
assert 'constraint_flux_QO_pbias_1' in content, "PBIAS constraint not found!"
print("Constraint block verified in ostIn.txt.")

### Check: eval.json contains constraints

In [ ]:
import json

with open('./wolf-creek-calibration-with-constraints/etc/eval/eval.json', 'r') as f:
    eval_config = json.load(f)

assert 'constraints' in eval_config, "constraints key not found in eval.json!"
assert 'flux' in eval_config['constraints'], "flux group not found in constraints!"

flux_constraints = eval_config['constraints']['flux']['QO']
print("Constraint metrics defined in eval.json:")
for metric_name, metric_info in flux_constraints.items():
    print(f"  {metric_name}: lower={metric_info['lower']}, "
          f"upper={metric_info['upper']}, "
          f"cost_factor={metric_info['cost_factor']}, "
          f"expressions={metric_info['expressions']}")

## 7. Running the calibration

To run the calibration with Ostrich, navigate to the output directory and execute:

```bash
cd wolf-creek-calibration-with-constraints/
ost
```

Ostrich will:
1. Sample parameter values using DDS
2. Run `eval.py` for each iteration, which computes both objective functions **and** constraint values
3. Read constraint CSVs and evaluate violations against the bounds
4. Add penalties to the objective function when constraints are violated
5. Iterate until convergence or max iterations

The constraint tracking output will be written to `OstGcopOut` in the calibration directory.

## Advanced: Using helper and custom constraint groups

Constraints support the same `helper` and `custom` groups as objective functions. This is useful when you need to:
- Compute intermediate values (helpers) that are combined into a final constraint (custom)
- Create composite constraints from multiple metrics

Example:
```python
'constraints': {
    'helper': {
        'QO': {
            'pbias_val': {
                'lower': -1e99, 'upper': 1e99,
                'cost_factor': 1,
                'expressions': ['alaska_72'],
            },
        },
    },
    'custom': {
        'QO': {
            'abs_pbias': {
                'lower': 0.0, 'upper': 25.0,
                'cost_factor': 100,
                'expressions': ["abs(helper_ofs['QO']['pbias_val'])"],
            },
        },
    },
}
```

Note: Constraint helpers are **isolated** from objective function helpers -- they cannot cross-reference each other.